# 01 - PreparaÃ§Ã£o e Amostragem de Dados

Fluxo para carregar os dados brutos, limpar cancelados/desviados, criar features bÃ¡sicas e salvar uma amostra estÃ¡vel para as demais anÃ¡lises.

## Objetivos
- Carregar CSVs brutos de voos, companhias e aeroportos.
- Remover voos cancelados ou desviados que nÃ£o possuem hora de chegada vÃ¡lida.
- Criar variÃ¡veis de atraso e tempo.
- Gerar amostra estÃ¡vel (`data/processed/flights_sample.parquet`) para acelerar EDA e modelagem.

In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/flights.csv")
AIRLINES = Path("../data/airlines.csv")
AIRPORTS = Path("../data/airports.csv")
OUT = Path("../data/processed/flights_sample.parquet")
OUT.parent.mkdir(parents=True, exist_ok=True)

### 1. Carregar dados brutos (colunas relevantes)

In [ ]:
cols = [
    "YEAR","MONTH","DAY","DAY_OF_WEEK","AIRLINE","FLIGHT_NUMBER","TAIL_NUMBER",
    "ORIGIN_AIRPORT","DESTINATION_AIRPORT","SCHEDULED_DEPARTURE","DEPARTURE_TIME",
    "DEPARTURE_DELAY","ARRIVAL_DELAY","DISTANCE","DIVERTED","CANCELLED","CANCELLATION_REASON"
]
flights_raw = pd.read_csv(RAW, usecols=cols, low_memory=False)
airlines = pd.read_csv(AIRLINES)
airports = pd.read_csv(AIRPORTS)
flights_raw.shape

### 2. Limpeza e enriquecimento
- Remove voos cancelados (`CANCELLED = 1`) e desviados (`DIVERTED = 1`).
- Define variÃ¡vel-alvo `DELAYED` (>15 min).
- Cria hora de partida (`DEP_HOUR`), perÃ­odo do dia e flag de fim de semana.

In [ ]:
flights = flights_raw[(flights_raw["CANCELLED"] == 0) & (flights_raw["DIVERTED"] == 0)].copy()
flights["DELAYED"] = (flights["ARRIVAL_DELAY"] > 15).astype(int)
flights["DEP_HOUR"] = (flights["SCHEDULED_DEPARTURE"] // 100).astype(int)
flights["IS_WEEKEND"] = flights["DAY_OF_WEEK"].isin([6,7]).astype(int)
flights["PERIOD_OF_DAY"] = pd.cut(
    flights["DEP_HOUR"], bins=[-1,5,11,17,23], labels=["madrugada","manh?","tarde","noite"]
)
flights[["ARRIVAL_DELAY","DELAYED","DEP_HOUR"]].head()

### 3. Amostragem reprodutÃ­vel
Escolhemos 300k voos para equilibrar volume e velocidade.

In [ ]:
sample = flights.sample(n=300_000, random_state=42) if len(flights) > 300_000 else flights
sample.shape

### 4. Salvar amostra processada

In [ ]:
sample.to_parquet(OUT, index=False)
print(f"Amostra salva em {OUT}")
sample.head()

### 5. Notas rÃ¡pidas
- `DELAYED` jÃ¡ usa atraso de chegada (>15 min).
- Colunas categÃ³ricas de mapeamento estÃ£o em `airlines` e `airports`.
- As demais etapas (EDA, modelagem, clusterizaÃ§Ã£o) consomem apenas a amostra para manter a execuÃ§Ã£o leve.